In [1]:
import numpy as np
import pickle
import optuna

In [11]:
# Load bounds
with open(r"D:\UST Project\UST_Analog_automation\notebooks\new_data_exp\trained_models\param_bounds.pkl", "rb") as f:
    bounds = pickle.load(f)

feature_names = list(bounds.keys())  # ['a','b','c','d']

# Log sampling config (optimizer-side only)
feature_config = {
    "a": {"log": True},
    "b": {"log": False},  # includes 0
    "c": {"log": True},
    "d": {"log": True},
}

# Load trained forward models
model_gain = pickle.load(open(r"D:\UST Project\UST_Analog_automation\notebooks\new_data_exp\trained_models\model_gain_xgboost.pkl", "rb"))
model_pm   = pickle.load(open(r"D:\UST Project\UST_Analog_automation\notebooks\new_data_exp\trained_models\model_pm_xgboost.pkl", "rb"))
model_ugf  = pickle.load(open(r"D:\UST Project\UST_Analog_automation\notebooks\new_data_exp\trained_models\model_ugf_xgboost.pkl", "rb"))


In [25]:
GAIN_TARGET = 55.10516431	   # example
PM_MAX      = 66.97095421    # constraint
UGF_TARGET  = 120.42698870000001	  # example

In [26]:
# ============================================================
# 4. INVERSE LOSS FUNCTION
# ============================================================
def inverse_loss(x):
    """
    x shape: (1, 4)
    """
    gain = model_gain.predict(x)[0]
    pm   = model_pm.predict(x)[0]
    ugf  = model_ugf.predict(x)[0]

    loss = (
        abs(gain - GAIN_TARGET) +
        max(0.0, PM_MAX - pm) +
        abs(ugf - UGF_TARGET)
    )
    return loss


In [27]:
def objective(trial):
    x = []

    for f in feature_names:
        low, high = bounds[f]

        if feature_config[f]["log"]:
            val = trial.suggest_float(f, low, high, log=True)
        else:
            val = trial.suggest_float(f, low, high)

        x.append(val)

    x = np.array(x).reshape(1, -1)
    return inverse_loss(x)

In [28]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=50
    )
)

study.optimize(objective, n_trials=200, show_progress_bar=True)

[I 2026-02-06 21:01:18,283] A new study created in memory with name: no-name-1821c08a-8adc-4fae-b6e5-ee596d47c6dd


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-02-06 21:01:18,327] Trial 0 finished with value: 156.96646118164062 and parameters: {'a': 6.59269988761102e-06, 'b': 0.0003698270095505816, 'c': 4.462395092148129e-06, 'd': 6.37710619087497e-05}. Best is trial 0 with value: 156.96646118164062.
[I 2026-02-06 21:01:18,331] Trial 1 finished with value: 117.59207153320312 and parameters: {'a': 5.478384061253937e-06, 'b': 6.068172801571452e-05, 'c': 2.521068595986405e-06, 'd': 7.999520174016835e-05}. Best is trial 1 with value: 117.59207153320312.
[I 2026-02-06 21:01:18,336] Trial 2 finished with value: 111.35490417480469 and parameters: {'a': 7.987991737671789e-06, 'b': 0.0002754395954973417, 'c': 2.44222604110033e-06, 'd': 8.734449411161188e-05}. Best is trial 2 with value: 111.35490417480469.
[I 2026-02-06 21:01:18,340] Trial 3 finished with value: 141.96209716796875 and parameters: {'a': 9.717639958417919e-06, 'b': 8.259972294864981e-05, 'c': 2.7997458439219924e-06, 'd': 4.485592596776692e-05}. Best is trial 2 with value: 111.35

In [29]:
best_params = study.best_params
x_best = np.array([best_params[f] for f in feature_names]).reshape(1, -1)

gain_best = model_gain.predict(x_best)[0]
pm_best   = model_pm.predict(x_best)[0]
ugf_best  = model_ugf.predict(x_best)[0]

print("\n===== INVERSE PREDICTION =====")
for k in feature_names:
    print(f"{k} = {best_params[k]:.6e}")

print("\nPredicted outputs:")
print(f"GAIN = {gain_best:.6f}")
print(f"PM   = {pm_best:.6f}")
print(f"UGF  = {ugf_best:.6f}")
print(f"Final loss = {study.best_value:.6f}")


===== INVERSE PREDICTION =====
a = 1.103786e-05
b = 2.894018e-04
c = 5.071227e-06
d = 7.787758e-05

Predicted outputs:
GAIN = 54.916481
PM   = 68.266685
UGF  = 120.716606
Final loss = 0.478302


In [24]:
import pandas as pd

df = pd.read_csv("Opam.csv")
pd.set_option('display.float_format', '{:.10f}'.format)


df.head()

,a,b,c,d,gain,pm,ugf
0,0.0000112000,0.0001320120,0.0000024000,0.0000384000,18.7427113100,87.2589267500,12.8594501900
1,0.0000112000,0.0001320120,0.0000024000,0.0000427000,20.0224857400,88.4842564900,14.2978920500
2,0.0000112000,0.0001320120,0.0000024000,0.0000469000,21.2202001200,89.6240725500,15.5129365100
3,0.0000112000,0.0001320120,0.0000024000,0.0000512000,22.3550526600,90.9081618400,17.4676334600
4,0.0000112000,0.0001320120,0.0000024000,0.0000555000,23.4437963200,92.1196945900,19.5720128900


In [14]:
print("\n===== STABILITY CHECK =====")
for i in range(5):
    noise = np.random.normal(0, 5e-8, size=x_best.shape)
    g = model_gain.predict(x_best + noise)[0]
    p = model_pm.predict(x_best + noise)[0]
    u = model_ugf.predict(x_best + noise)[0]
    print(f"{i+1}: GAIN={g:.4f}, PM={p:.4f}, UGF={u:.4f}")


===== STABILITY CHECK =====
1: GAIN=18.7872, PM=86.4662, UGF=12.0975
2: GAIN=18.7872, PM=86.4662, UGF=12.0975
3: GAIN=18.7872, PM=86.4662, UGF=12.0975
4: GAIN=18.7872, PM=86.4662, UGF=12.0975
5: GAIN=18.7872, PM=86.4662, UGF=12.0975


In [ ]:
print("\n===== TOP 5 INVERSE SOLUTIONS =====")
top_trials = sorted(study.trials, key=lambda t: t.value)[:5]

for i, t in enumerate(top_trials):
    print(f"\nSolution {i+1} | Loss = {t.value:.6f}")
    for f in feature_names:
        print(f"{f} = {t.params[f]:.6e}")

In [39]:
import pandas as pd 

df = pd.read_csv("Opam.csv")

df.head()

,a,b,c,d,gain,pm,ugf
0,0.0000048000000000,0.0000448000000000,0.0000024000000000,0.0000384000000000,18.9109775499999984,86.2353259100000002,12.1724450399999995
1,0.0000048000000000,0.0000448000000000,0.0000024000000000,0.0000427000000000,20.1793160600000014,87.4904806299999933,13.6422904700000007
2,0.0000048000000000,0.0000448000000000,0.0000024000000000,0.0000469000000000,21.3631409199999993,88.6525631399999980,14.8823551999999992
3,0.0000048000000000,0.0000448000000000,0.0000024000000000,0.0000512000000000,22.4827260200000012,89.7777121200000039,16.0733908000000021
4,0.0000048000000000,0.0000448000000000,0.0000024000000000,0.0000555000000000,23.5539255099999991,91.0271390599999961,18.2064033899999984


In [40]:
pd.set_option('display.float_format', '{:.16f}'.format)
